# Second-round forward selection within the MiSTIC unified set

After the established first-stage MiSTIC search, this experiment starts a new forward search restricted to the complete feature union discovered by the member set. Every unified feature is evaluated as a singleton, then exactly one remaining feature is added per step. The knee of this second CV curve defines one final unified predictor. Comparisons use the same fixed blind classification and harder-regression sets as the previous analyses.

In [ ]:
from pathlib import Path
import sys
repo_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'mistic' / 'svmSet.py').exists())
example_dir = repo_root / 'mistic' / 'examples'
if str(example_dir) not in sys.path: sys.path.insert(0, str(example_dir))
from synthetic100_second_round_forward_selection import run_experiment
results, curves = run_experiment()
results

In [ ]:
classification = results.query("task == 'classification'").groupby('method')[['roc_auc', 'f1', 'balanced_accuracy', 'accuracy', 'num_features', 'signal_recall']].agg(['mean', 'std'])
classification.round(4)

In [ ]:
regression = results.query("task == 'regression'").groupby('method')[['r2', 'rmse', 'mae', 'num_features', 'signal_recall']].agg(['mean', 'std'])
regression.round(4)

In [ ]:
import pandas as pd
paired = []
for task, metric in [('classification', 'roc_auc'), ('regression', 'r2')]:
    wide = results.query('task == @task').pivot(index='seed', columns='method', values=metric)
    baseline = wide['MiSTIC unified default']
    for method in wide.columns.drop('MiSTIC unified default'):
        delta = wide[method] - baseline
        paired.append({'task': task, 'method_vs_default': method, 'mean_difference': delta.mean(),
                       'wins': (delta > 0).sum(), 'ties': (delta == 0).sum(), 'losses': (delta < 0).sum()})
pd.DataFrame(paired)

In [ ]:
results.query("method == 'second-round forward knee'").groupby(['task', 'second_round_rule']).size().rename('count').to_frame()